# Part 3 — Measuring Similarity

*Turning "close in embedding space" into a single number you can rank by.*

📖 Read the essay: https://www.mefby.com/essays/measuring-similarity

🐍 Companion script: `similarity.py`

**What you'll build**

- **Euclidean distance** — the straight-line gap between two arrowheads.
- The **dot product** — overlap, blended with length, and the fast one at scale.
- **Cosine similarity** — direction only, the default metric in RAG.
- The **'aha'**: cosine similarity *is* the dot product of normalized vectors.
- **Top-k retrieval** — the ranking function at the heart of every RAG system.

> Runs fully offline. This part is arithmetic a calculator can do: pure NumPy and the standard library, no model, no API key, no network. NumPy is used for the one-liner but a transparent pure-Python fallback covers it if NumPy is missing, so every cell runs either way.

## Setup

Everything here is multiplies, adds, and one square root. We lead with pure Python (so you can see every operation), then show the NumPy one-liner the essay gives as the "in code" version. NumPy is optional: if it is missing we route transparently back to the pure-Python math so the same numbers still print.

In [ ]:
import math

# NumPy is nice for the one-liner cosine, but this whole part is just
# multiplies, adds, and one square root. If NumPy is missing we fall back to
# a transparent pure-Python stand-in so the demo always runs. (Later parts
# use this same "works without the heavy library" pattern.)
try:
    import numpy as np
    HAVE_NUMPY = True
except ImportError:  # exercised only on a NumPy-less box
    np = None
    HAVE_NUMPY = False

Vector = list  # a vector is just an ordered list of numbers (Part 2)

print(f"NumPy available: {HAVE_NUMPY}  (falls back to pure Python if not)")

## The debt from Part 2: turning "close" into a number

**Part 2 — Embeddings** ended on a debt. It kept saying two embeddings were "close" or "near" without ever defining how a machine *measures* closeness. This part pays that debt.

Here is the exact moment we have to handle. A user asks *"What is our refund window?"* and we embed it into a query vector. Sitting in our vector store are the embeddings of every chunk, including one that reads *"Refunds are accepted within 30 days of purchase."* To pick the best chunk, retrieval has to *rank*. To rank, it needs a single number per chunk: **how similar is this chunk's vector to the query's?** That number is a **score**, and the function that produces it is a **similarity metric**.

We'll use tiny 2-D vectors so the arithmetic stays on paper. Real embeddings have hundreds of dimensions, but the procedure is *identical* — just with more terms to add.

In [ ]:
# The three tiny vectors the whole chapter turns on.
A = [3, 4]
B = [4, 3]
C = [6, 8]   # exactly A doubled: same direction, twice as long

print(f"A = {A}")
print(f"B = {B}")
print(f"C = {C}   (A doubled: same direction, twice as long)")

## Similarity vs. distance: same idea, opposite direction

One confusion trips up almost everyone, because the two words point opposite ways:

- A **similarity** score goes *up* as two vectors become more alike. Higher is better; a perfect match is the maximum.
- A **distance** goes *down* as two vectors become more alike. Lower is better; a perfect match is 0.

They measure the same underlying thing, but the direction of "good" flips. "Sort by similarity" keeps the *largest* scores; "sort by distance" keeps the *smallest*. Keep that straight and half the confusion here disappears.

## Euclidean distance: the straight-line gap

The most intuitive measure of "how far apart" is the one you use in the physical world: the straight-line distance between two points. That is **Euclidean distance** — the Pythagorean theorem, allowed to run in as many dimensions as the embedding has.

Take the difference between the two vectors component by component, square each difference, add them up, take the square root:

$$\text{euclidean}(A, B) = \sqrt{(A_1 - B_1)^2 + \dots + (A_n - B_n)^2}$$

It is a *distance* — smaller means more similar, identical vectors score 0. Its weakness for text: it cares about **magnitude** (length). A one-line note and a three-paragraph summary about the same topic can have quite different magnitudes, which pushes their arrowheads far apart, and Euclidean dutifully reports "not very similar."

In [ ]:
def euclidean(a: Vector, b: Vector) -> float:
    # difference per component, square each, sum, square-root.
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))

# euclidean(A, B) = sqrt((3-4)^2 + (4-3)^2) = sqrt(1 + 1) = sqrt(2) ~= 1.41
print(f"euclidean(A, B) = {euclidean(A, B):.4f}   (~ sqrt(2))")

## Dot product: overlap, blended with length

The recipe is even simpler: multiply the two vectors component by component, then add up the results.

$$A \cdot B = A_1 B_1 + A_2 B_2 + \dots + A_n B_n$$

The dot product is a *similarity* — bigger means more aligned. It quietly blends two things at once: how much the vectors point the same way (their angle) **and** how long they are (their magnitudes). Same direction → big positive; right angles → 0; opposite → negative.

Its great strength is **speed**: just multiplies and adds, the operation hardware is most optimized for. It is what runs under the hood when a vector store scores a query against millions of chunks (we'll lean on that in **Part 4**). Its mirror-image weakness: it is **unnormalized**, so a longer vector scores higher regardless of direction.

In [ ]:
def dot(a: Vector, b: Vector) -> float:
    # multiply matching components, add them up.
    return sum(ai * bi for ai, bi in zip(a, b))

# A . B = (3*4) + (4*3) = 12 + 12 = 24
print(f"dot(A, B) = {dot(A, B)}")
# A . C = (3*6) + (4*8) = 18 + 32 = 50  -- C scores high purely for being long
print(f"dot(A, C) = {dot(A, C)}   (jumps to 50: C rewarded for length)")

## Magnitude (norm): the length of a vector

Before cosine we need one helper: the **magnitude** (or **norm**), written $\lVert A \rVert$. It is the length of the vector — the distance from the origin to its tip. Square the components, add, square-root: exactly Euclidean distance measured from the origin.

$$\lVert A \rVert = \sqrt{A_1^2 + A_2^2 + \dots + A_n^2}$$

In [ ]:
def magnitude(a: Vector) -> float:
    # square the components, add, square-root.
    return math.sqrt(sum(ai * ai for ai in a))

# ||A|| = sqrt(3^2 + 4^2) = sqrt(9 + 16) = sqrt(25) = 5  (the classic 3-4-5)
print(f"magnitude(A) = {magnitude(A)}")
print(f"magnitude(B) = {magnitude(B)}")
print(f"magnitude(C) = {magnitude(C)}   (twice as long as A)")

## Cosine similarity: direction only

The star of the chapter. Throw away length completely and measure only the **angle** between the two vectors — cosine similarity is the cosine of that angle. It is the dot product divided by *both* magnitudes, and that division is exactly the step that cancels magnitude out. What survives is pure direction.

$$\cos(A, B) = \frac{A \cdot B}{\lVert A \rVert \, \lVert B \rVert}$$

It runs from `-1` (pointing exactly opposite) through `0` (right angles, unrelated) to `1` (exactly the same direction); for typical text embeddings it lands between `0` and `1`. Why is "direction only" right for text? Because that is where embedding models put the meaning: two passages about the same idea are trained to point the *same way* even if their lengths differ. That is why **cosine similarity is the default metric in RAG**.

In [ ]:
def cosine(a: Vector, b: Vector) -> float:
    # dot product divided by BOTH magnitudes -- the division cancels length.
    return dot(a, b) / (magnitude(a) * magnitude(b))

# cosine(A, B) = 24 / (5 * 5) = 24 / 25 = 0.96  -> nearly the same direction
print(f"cosine(A, B) = {cosine(A, B):.2f}")
# cosine(A, C) = 50 / (5 * 10) = 50 / 50 = 1.00  -> identical direction
print(f"cosine(A, C) = {cosine(A, C):.2f}   (length ignored: A and C point the same way)")

### Three metrics, three different verdicts

Lay the three side by side on the same vectors and you can *see* why each one votes the way it does. Watch the `A,C` row: `A` and `C` point the exact same way (`C` is just `A` doubled), yet Euclidean calls them far apart and the dot product rewards `C` for being long. Only cosine reports the truth — same direction, score `1.00`.

In [ ]:
print(f"{'pair':>6} | {'euclidean':>9} | {'dot':>5} | {'cosine':>7}")
print('-' * 38)
for name, x, y in [('A,B', A, B), ('A,C', A, C), ('B,C', B, C)]:
    print(
        f"{name:>6} | {euclidean(x, y):>9.2f} | "
        f"{dot(x, y):>5.0f} | {cosine(x, y):>7.2f}"
    )

## Normalization: rescale to length 1

**Normalization** rescales a vector to length 1 while keeping its direction. The result is a **unit vector**, written $\hat{A} = A / \lVert A \rVert$. Normalize `A` and `C` — same direction, different lengths — and they collapse to the *same* unit vector.

In [ ]:
def normalize(a: Vector) -> Vector:
    mag = magnitude(a)
    return [ai / mag for ai in a]

# A / 5 = [0.6, 0.8]   and   C / 10 = [0.6, 0.8]  -- the SAME unit vector.
print(f"normalize(A) = {[round(v, 2) for v in normalize(A)]}")
print(f"normalize(C) = {[round(v, 2) for v in normalize(C)]}   (same unit vector!)")

## The 'aha': cosine is the dot product of normalized vectors

Here is the insight that ties the chapter together. Look at the cosine formula: it is the dot product divided by the two magnitudes. Now suppose we first normalize each vector. Once both are unit length, their magnitudes are both 1, so dividing by them does *nothing*, and the formula collapses:

$$\cos(A, B) = \hat{A} \cdot \hat{B}$$

In words: **cosine similarity is just the dot product of normalized vectors.** This is load-bearing in real systems. The dot product is the fast one, so many vector databases **normalize every embedding once, when it is stored**, then run the cheap dot product at query time — getting cosine's meaning-focused, length-blind behavior at the dot product's speed. When you read that a database "uses inner product on normalized vectors," that now means something precise: it is computing cosine similarity the efficient way.

In [ ]:
def cosine_via_dot(a: Vector, b: Vector) -> float:
    # normalize first, then a plain dot product == cosine.
    return dot(normalize(a), normalize(b))

# (0.6*0.6) + (0.8*0.8) = 0.36 + 0.64 = 1.00 = the cosine.
print(f"cosine(A, C)         = {cosine(A, C):.2f}")
print(f"cosine_via_dot(A, C) = {cosine_via_dot(A, C):.2f}   (same number, the efficient way)")
print(f"cosine(A, B)         = {cosine(A, B):.2f}")
print(f"cosine_via_dot(A, B) = {cosine_via_dot(A, B):.2f}   (matches for A,B too)")

## The NumPy one-liner (what real systems lean on)

The essay's "if you'd rather see it in code" version. In NumPy, `@` is the dot product and `np.linalg.norm` is the magnitude, so cosine is one line. This is the **headline** — the path you'd actually ship. To keep the notebook runnable even on a box without NumPy, we wrap it: if NumPy is missing we route transparently back to the pure-Python `cosine` above and print the same number. That is the same "works without the heavy library" guard the rest of the series uses.

In [ ]:
def cosine_numpy(a: Vector, b: Vector) -> float:
    if not HAVE_NUMPY:
        # Transparent stand-in: identical math, no dependency.
        print('[NumPy unavailable] -> offline pure-Python fallback')
        return cosine(a, b)
    a_arr, b_arr = np.array(a, dtype=float), np.array(b, dtype=float)
    return float(a_arr @ b_arr / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))

# The essay's two-line snippet, made into a reusable function.
print(f"cosine_numpy(A, B) = {cosine_numpy(A, B):.2f}   (the @ one-liner)")
print(f"cosine_numpy(A, C) = {cosine_numpy(A, C):.2f}")

## Check it against the paper

A metric you can't compute is a metric you don't trust. The essay worked these numbers by hand; let's assert that our code agrees. If every line passes, the implementations match the arithmetic you can check yourself.

In [ ]:
# --- A vs B ---------------------------------------------------------
assert dot(A, B) == 24
assert magnitude(A) == 5 and magnitude(B) == 5
assert math.isclose(cosine(A, B), 0.96)
assert math.isclose(cosine_numpy(A, B), 0.96)
assert math.isclose(euclidean(A, B), math.sqrt(2))

# --- A vs C: the example that makes cosine earn its keep -------------
assert math.isclose(cosine(A, C), 1.00)   # identical direction
assert euclidean(A, C) == 5               # far apart by straight line
assert dot(A, C) == 50                    # rewarded purely for length

# --- The 'aha' made concrete ----------------------------------------
assert all(math.isclose(x, y) for x, y in zip(normalize(A), [0.6, 0.8]))
assert all(math.isclose(x, y) for x, y in zip(normalize(C), [0.6, 0.8]))
assert math.isclose(cosine_via_dot(A, C), 1.00)
assert math.isclose(cosine(A, B), cosine_via_dot(A, B))

print('All worked examples from the essay checked out.')

## Top-k retrieval: the RAG ranking function

Now the payoff. **Top-k retrieval** scores the query against *every* stored vector with cosine similarity, then keeps the `k` highest. Because cosine is a similarity (higher is better), we sort descending and take the first `k`.

Each stored item is a `(label, vector)` pair so we can show *which* chunk won — the way a real store keeps the text beside the vector. This is the brute-force / exact search the essay describes: fine for hundreds or thousands of chunks. Making it fast for *millions* is exactly what **Part 4 — Vector Databases & Indexing** is about.

In [ ]:
def top_k(query: Vector, store: list, k: int = 3):
    scored = [(label, cosine(query, vec)) for label, vec in store]
    # A SIMILARITY -> higher is better, so sort descending and take the first k.
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:k]

A query and a handful of "chunk" vectors. The winner is whichever vector points most nearly the *same way* as the query — length is ignored entirely.

In [ ]:
query = [1.0, 0.0]   # points straight along the x-axis
store = [
    ('chunk_0  same direction, far away', [9.0, 0.0]),   # cos 1.00 (length ignored!)
    ('chunk_1  45 degrees off',           [1.0, 1.0]),   # cos ~0.71
    ('chunk_2  near-aligned',             [4.0, 1.0]),   # cos ~0.97
    ('chunk_3  right angle, unrelated',   [0.0, 5.0]),   # cos 0.00
    ('chunk_4  pointing opposite',        [-2.0, 0.0]),  # cos -1.00
]

print(f"query = {query}")
results = top_k(query, store, k=3)
for rank, (label, score) in enumerate(results, start=1):
    print(f"  {rank}. score={score:>6.2f}  {label}")

print()
print('Note chunk_0 ties for first despite being 9x longer than the query:')
print('cosine ignores length and rewards pure direction -- the whole point.')

## The full headline demo

Finally, the companion script's demo end to end, so you see the real output the way `python3 similarity.py` prints it: the three vectors, the three metrics side by side, the 'aha', and top-k retrieval — all from the functions we built above.

In [ ]:
print(f"NumPy available: {HAVE_NUMPY}  (falls back to pure Python if not)\n")

print('Three tiny 2-D vectors (real embeddings just have more dimensions):')
print(f"  A = {A}")
print(f"  B = {B}")
print(f"  C = {C}   (A doubled: same direction, twice as long)\n")

print('Pairwise scores -- three metrics, three different verdicts:')
print(f"{'pair':>6} | {'euclidean':>9} | {'dot':>5} | {'cosine':>7}")
print('-' * 38)
for name, x, y in [('A,B', A, B), ('A,C', A, C), ('B,C', B, C)]:
    print(
        f"{name:>6} | {euclidean(x, y):>9.2f} | "
        f"{dot(x, y):>5.0f} | {cosine(x, y):>7.2f}"
    )

print("\nThe 'aha': cosine == dot product of normalized (unit-length) vectors")
print(f"  normalize(A) = {[round(v, 2) for v in normalize(A)]}")
print(f"  normalize(C) = {[round(v, 2) for v in normalize(C)]}   (same unit vector!)")
print(f"  cosine(A, C)         = {cosine(A, C):.2f}")
print(f"  cosine_via_dot(A, C) = {cosine_via_dot(A, C):.2f}")
print(f"  cosine_numpy(A, C)   = {cosine_numpy(A, C):.2f}   (the @ one-liner)")

print('\nTop-k retrieval (the RAG ranking function):')
print(f"  query = {query}")
for rank, (label, score) in enumerate(top_k(query, store, k=3), start=1):
    print(f"  {rank}. score={score:>6.2f}  {label}")

## What you learned

- Retrieval has to **rank**, and ranking needs one **similarity** score per chunk. A similarity goes *up* as vectors get more alike; a **distance** goes *down*. Same idea, opposite direction.
- **Euclidean distance** is the intuitive straight-line gap, but it is sensitive to **magnitude**, so passages of different lengths about the same topic can look far apart.
- The **dot product** is fast (just multiplies and adds) and is what runs at scale, but it is **unnormalized**: longer vectors score higher regardless of direction.
- **Cosine similarity** measures the *angle* only and ignores length, which is exactly right for text. It is the **default metric in RAG**.
- The 'aha': **cosine similarity is the dot product of normalized (unit-length) vectors.** Normalize once at storage time, then use the cheap dot product, and you get cosine's behavior at the dot product's speed.
- **Top-k retrieval** returns the `k` highest-scoring chunks — the ranking function at the heart of RAG. Match the metric to what the model was trained for.

### Next

We can now score *one* chunk against the query, and we know we'd have to do it for *every* chunk. **Part 4 — Vector Databases & Indexing** makes that fast at scale: how a vector database stores millions of embeddings and uses approximate nearest-neighbor search to find the closest ones without comparing against all of them.

📖 Next essay: https://www.mefby.com/essays/vector-databases-and-indexing